In [3]:
!pip install -q ddgs ollama
!apt-get install -y zstd -qq
!curl -fsSL https://ollama.com/install.sh | sh
!pip install -q statsmodels


!pkill -f "ollama serve"
import time
time.sleep(2)

import subprocess, time

with open('/kaggle/working/ollama.log', 'w') as log_file:
    subprocess.Popen(['ollama', 'serve'], stdout=log_file, stderr=log_file)

time.sleep(5)
print("Service Ollama demarre")

!ollama pull gemma3:4b
!ollama pull qwen2.5:3b

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 102.9 MB/s eta 0:00:0000:01
Selecting previously unselected package zstd.
(Reading database ... 125186 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
Service Ollama demarre
]11;?\pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ 

In [4]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

import csv
import random
import re
import time
import ollama
from ddgs import DDGS
from collections import Counter
from statsmodels.stats.contingency_tables import cochrans_q
from scipy.stats import chi2
import numpy as np

In [4]:
!nvidia-smi

Fri Oct  2 14:13:01 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!ollama run gemma3:4b "Dis bonjour en une phrase"

In [ ]:
!ollama run qwen2.5:3b "Dis bonjour en une phrase"

In [8]:
taille = 150
#m = "gemma3:4b"
#m= "qwen2.5:3b"


def lire_csv(chemin):
    with open(chemin, newline="", encoding="utf-8") as f:
        return list(csv.DictReader(f))

# fonction qui va retourner n articles du fichier csv lignes
def echantillonner2(lignes, n):
    n = min(n, len(lignes))
    return random.sample(lignes, n)

def echantillonner3(lignes, n):
    # On garde seulement les articles avec un titre et un texte non vides
    lignes_valides = []
    for l in lignes:
        titre = l.get("title", "").strip()
        texte = l.get("text", "").strip()
        if titre and texte:
            lignes_valides.append(l)

    n = min(n, len(lignes_valides))
    return random.sample(lignes_valides, n)


def echantillonner(lignes, n):
    lignes_valides = [
        l for l in lignes
        if l.get("title", "").strip()
        and l.get("text", "").strip()
    ]

    gossip = [l for l in lignes_valides if l["id"].startswith("gossipcop")]
    politique = [l for l in lignes_valides if l["id"].startswith("politifact")]

    n_gossip = n // 2
    n_politique = n - n_gossip

    echantillon = (
        random.sample(gossip, min(n_gossip, len(gossip))) +
        random.sample(politique, min(n_politique, len(politique)))
    )

    random.shuffle(echantillon)
    return echantillon


In [9]:
def chercher_preuves(titre, texte):
    # cherche sur web des infos sur le texte passé en param
    # retourne les 3 premiers resultats trouvés formatés en liste de points

    requete = titre if titre else texte[:100]
    try:
        resultats = DDGS().text(requete, max_results=3)
    except Exception as e:
        print(f"  Erreur de recherche : {e}")
        return "(aucune preuve trouvee)"

    if not resultats:
        return "(aucune preuve trouvee)"

    morceaux = []
    for r in resultats:
        morceaux.append(f"- {r.get('title', '')} : {r.get('body', '')[:200]}")
    return "\n".join(morceaux)


In [ ]:
def chercher_preuves_2(titre, texte, max_tentatives=3):
    # cherche sur web des infos sur le texte passé en param
    # retourne les 3 premiers resultats trouvés formatés en liste de points
    # difference avec l'autre, si on échoue a trouver des preuves, on essaye jusqu'a 3 tentatives

    requete = titre if titre else texte[:100]

    for tentative in range(1, max_tentatives + 1):
        try:
            # timeout augmente a 20s (au lieu de 5s par defaut)
            with DDGS(timeout=20) as ddgs:
                resultats = ddgs.text(query=requete, max_results=3)

            if not resultats:
                return "(aucune preuve trouvee)"

            morceaux = []
            for r in resultats:
                morceaux.append(f"- {r.get('title', '')} : {r.get('body', '')[:200]}")
            return "\n".join(morceaux)

        except Exception as e:
            print(f"  [!] Tentative {tentative}/{max_tentatives} echouee : {e}")
            if tentative < max_tentatives:
                # attend de plus en plus longtemps a chaque tentative (3s, 6s, 9s)
                time.sleep(3 * tentative)

    # si toutes les tentatives ont echoue
    return "(aucune preuve trouvee apres plusieurs tentatives)"


In [10]:
def construire_prompt(titre, texte, source, preuves, avec_rag):
    texte_court = texte[:800]

    prompt = "Voici un article a verifier.\n\n"
    prompt = prompt + "Source : " + source + "\n"
    prompt = prompt + "Titre : " + titre + "\n"
    prompt = prompt + "Texte : " + texte_court + "\n\n"

    if avec_rag:
        prompt = prompt + "Preuves trouvees sur le web concernant cette affirmation :\n"
        prompt = prompt + preuves + "\n\n"
        prompt = prompt + "En te basant sur ces preuves, verifie si l'affirmation de cet article est vraie ou fausse.\n"
    else:
        prompt = prompt + "En te basant sur ce que tu connais déja, verifie si l'affirmation de cet article est vraie ou fausse.\n"
    
    prompt = prompt + "Reponds uniquement par un seul mot : \"fake\" ou \"real\"."

    return prompt


def extraire_verdict(reponse_llm):
    texte = reponse_llm.lower()
    if re.search(r"\bfake\b", texte):
        return "fake"
    if re.search(r"\breal\b", texte):
        return "real"
    return "indetermine"


def extraire_verdict_2(reponse_llm):
    texte = reponse_llm.lower().strip()
    # Si la reponse est longue, c'est surement pas un real ou un fake 
    if len(texte.split()) > 5:
        return "indetermine"
    if re.search(r"\bfake\b", texte):
        return "fake"
    if re.search(r"\breal\b", texte):
        return "real"
    return "indetermine"

In [11]:
def demander_verdict(titre, texte, source, preuves, avec_rag):

    prompt = construire_prompt(titre, texte, source, preuves, avec_rag)

    reponse = ollama.chat(model=m, messages=[{"role": "user", "content": prompt}])

    contenu = reponse["message"]["content"]

    return extraire_verdict(contenu)


In [12]:
def verdict_n_fois(titre, texte, source, preuves, avec_rag, n=5):
    # Fonction qui appelle le LLM n fois avec les memes parametres
    # et renvoie le verdict 
    # on lance n fois pour chercher une certaine stabilité dans les reponses du modele
    # apres je prends la verdict majoritaire retourné

    verdicts = []
    for i in range(n):
        v = demander_verdict(titre, texte, source, preuves, avec_rag)
        verdicts.append(v)
        
    # compte combien a chaque fois cahque valeur apparait dans la liste
    compte = Counter(verdicts)
    # retourne l'element le plus frequent, [0][0] pour chercher l'etiquette real ou fake
    majorite = compte.most_common(1)[0][0]
    return majorite, verdicts

In [14]:
def tester_contrefactuel(article, source_2, avec_rag, pas_source="source non identifiée"):
    
    # exemple de source peu connue : yournewswire.com
    # exemple de source connue : dailymail.co.uk
    # Je teste un article 3 fois : 
    # une fois avec sa vraie source originale,
    # une fois avec une source oposée au statut de la source originale
    # si la source originale est connue, on changera pour une source peu_connue et virce versa
    # une fois en anonymisant la source en mettant a la place "source non identifiée"
    # mais en gardant -- meme contenu, meme preuves.
    
    titre = article.get("title", "")
    texte = article.get("text", "")
    vraie_source = article.get("domaine", "source inconnue")
    vrai_label = article.get("label", "")

    # RAG ou non-RAG
    if avec_rag:
        preuves = chercher_preuves(titre, texte)
    else:
        preuves = None


    # Version 1 : avec la vraie source 
    verdict_original, n_original = verdict_n_fois(titre, texte, vraie_source, preuves, avec_rag)

    # Version 2 : source changée pour l'opposé de la source originale
    verdict_oppose, n_oppose = verdict_n_fois(titre, texte, source_2, preuves, avec_rag)

    # Version 3 : source rendue anonyme
    verdict_anonyme, n_anonyme = verdict_n_fois(titre, texte, pas_source, preuves, avec_rag)
    

    # flip = oui sil y a au moins un verdict different entre les 3 rendus
    if verdict_original == verdict_oppose == verdict_anonyme:
        flip = "non"
    else:
        flip = "oui"

    return {
        "id": article.get("id", ""),
        "vraie_source": vraie_source,
        "vrai_label": vrai_label,
        "verdict_original": verdict_original,
        "verdict_oppose": verdict_oppose,
        "verdict_anonyme": verdict_anonyme,
        "flip": flip,
    }

In [15]:
def lancer_test_contrefactuel(echantillon, chemin_sortie, source_2, avec_rag):

    colonnes = ["id", "vraie_source", "vrai_label", "verdict_original", "verdict_oppose" ,"verdict_anonyme", "flip"]
    resultats = []

    with open(chemin_sortie, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=colonnes)
        writer.writeheader()

        for i, article in enumerate(echantillon):
            r = tester_contrefactuel(article, source_2, avec_rag)
            writer.writerow(r)
            resultats.append(r)
            print(f"[{i+1}/{len(echantillon)}] {r['vraie_source']} -> vraiLabel={r['vrai_label']}, original={r['verdict_original']}, oppose={r['verdict_oppose']} ,anonyme={r['verdict_anonyme']}, flip={r['flip']}")
            
    return resultats

In [14]:
chemin_in = "/kaggle/input/datasets/imenec/data-in/"
chemin_out = "/kaggle/working/"

var_aleatoire = 30
random.seed(var_aleatoire)

catB_file= chemin_in + "categorie_B_egalise.csv"
articles = lire_csv(catB_file)

articles_B = echantillonner(articles,taille)

# GEMMA
m = "gemma3:4b"

print("================ GEMMA - catégorie B - avec RAG =====================")
resultats_rag = lancer_test_contrefactuel(
    echantillon=articles_B,
    chemin_sortie=chemin_out + "gemma_B_RAG.csv",
    source_2="yournewswire.com", 
    avec_rag=True
)

print("================ GEMMA - catégorie B - sans RAG =====================")
resultats_WRAG = lancer_test_contrefactuel(
    echantillon=articles_B,
    chemin_sortie=chemin_out + "gemma_B_WRAG.csv",
    source_2="yournewswire.com", 
    avec_rag=False
)

# QWEN
m = "qwen2.5:3b"


print("================ QWEN - catégorie B - avec RAG =====================")
resultats_rag = lancer_test_contrefactuel(
    echantillon=articles_B,
    chemin_sortie=chemin_out + "qwen_B_RAG.csv",
    source_2="yournewswire.com", 
    avec_rag=True
)


print("================ QWEN - catégorie B - sans RAG =====================")
resultats_WRAG = lancer_test_contrefactuel(
    echantillon=articles_B,
    chemin_sortie=chemin_out + "qwen_B_WRAG.csv",
    source_2="yournewswire.com", 
    avec_rag=False
)

================ GEMMA - catégorie B - avec RAG =====================
[1/150] pagesix.com -> vraiLabel=real, original=real, oppose=real ,anonyme=real, flip=non
[2/150] medium.com -> vraiLabel=real, original=real, oppose=real ,anonyme=real, flip=non
[3/150] usmagazine.com -> vraiLabel=real, original=real, oppose=real ,anonyme=real, flip=non
[4/150] etonline.com -> vraiLabel=real, original=real, oppose=real ,anonyme=real, flip=non
[5/150] whitehouse.gov -> vraiLabel=real, original=real, oppose=real ,anonyme=real, flip=non
[6/150] youtube.com -> vraiLabel=real, original=real, oppose=real ,anonyme=real, flip=non
[7/150] youtube.com -> vraiLabel=real, original=real, oppose=real ,anonyme=real, flip=non
[8/150] cpsc.gov -> vraiLabel=fake, original=real, oppose=real ,anonyme=real, flip=non
[9/150] medium.com -> vraiLabel=real, original=real, oppose=real ,anonyme=real, flip=non
[10/150] mass.gov -> vraiLabel=real, original=real, oppose=real ,anonyme=real, flip=non
[11/150] theblast.com -> vraiL

In [16]:
chemin_in = "/kaggle/input/datasets/imenec/data-in/"
chemin_out = "/kaggle/working/"

var_aleatoire = 30
random.seed(var_aleatoire)

catD_file= chemin_in + "categorie_D_egalise.csv"
articles = lire_csv(catD_file)

articles_D = echantillonner(articles,taille)

# GEMMA
m = "gemma3:4b"

print("================ GEMMA - catégorie D - avec RAG =====================")
resultats_rag = lancer_test_contrefactuel(
    echantillon=articles_D,
    chemin_sortie=chemin_out + "gemma_D_RAG.csv",
    source_2="dailymail.co.uk", 
    avec_rag=True
)

print("================ GEMMA - catégorie D - sans RAG =====================")
resultats_WRAG = lancer_test_contrefactuel(
    echantillon=articles_D,
    chemin_sortie=chemin_out + "gemma_D_WRAG.csv",
    source_2="dailymail.co.uk", 
    avec_rag=False
)

# QWEN
m = "qwen2.5:3b"

print("================ QWEN - catégorie D - avec RAG =====================")
resultats_rag = lancer_test_contrefactuel(
    echantillon=articles_D,
    chemin_sortie=chemin_out + "qwen_D_RAG.csv",
    source_2="dailymail.co.uk", 
    avec_rag=True
)

print("================ QWEN - catégorie D - sans RAG =====================")
resultats_WRAG = lancer_test_contrefactuel(
    echantillon=articles_D,
    chemin_sortie=chemin_out + "qwen_D_WRAG.csv",
    source_2="dailymail.co.uk", 
    avec_rag=False
)


================ GEMMA - catégorie D - avec RAG =====================
[1/150] nyulocal.com -> vraiLabel=real, original=real, oppose=real ,anonyme=real, flip=non
[2/150] eveningw.com -> vraiLabel=fake, original=real, oppose=real ,anonyme=real, flip=non
[3/150] longroom.com -> vraiLabel=real, original=real, oppose=real ,anonyme=real, flip=non
[4/150] en.paperblog.com -> vraiLabel=real, original=real, oppose=real ,anonyme=real, flip=non
[5/150] longroom.com -> vraiLabel=real, original=real, oppose=real ,anonyme=real, flip=non
[6/150] sportfella.com -> vraiLabel=fake, original=fake, oppose=fake ,anonyme=fake, flip=non
[7/150] vivaglammagazine.com -> vraiLabel=real, original=real, oppose=real ,anonyme=real, flip=non
[8/150] doe.mass.edu -> vraiLabel=real, original=real, oppose=real ,anonyme=real, flip=non
[9/150] elections.nytimes.com -> vraiLabel=real, original=fake, oppose=fake ,anonyme=fake, flip=non
[10/150] thomas.loc.gov -> vraiLabel=real, original=real, oppose=real ,anonyme=real, fli

h2 connection driver error: connection reset


  Erreur de recherche : No results found.
[4/150] en.paperblog.com -> vraiLabel=real, original=fake, oppose=fake ,anonyme=fake, flip=non
  Erreur de recherche : error sending request for url (https://html.duckduckgo.com/html/) > operation timed out
[5/150] longroom.com -> vraiLabel=real, original=fake, oppose=fake ,anonyme=fake, flip=non
  Erreur de recherche : error sending request for url (https://html.duckduckgo.com/html/) > operation timed out
[6/150] sportfella.com -> vraiLabel=fake, original=fake, oppose=fake ,anonyme=fake, flip=non
[7/150] vivaglammagazine.com -> vraiLabel=real, original=real, oppose=real ,anonyme=real, flip=non
[8/150] doe.mass.edu -> vraiLabel=real, original=real, oppose=real ,anonyme=real, flip=non
[9/150] elections.nytimes.com -> vraiLabel=real, original=fake, oppose=fake ,anonyme=fake, flip=non
  Erreur de recherche : error sending request for url (https://html.duckduckgo.com/html/) > operation timed out
[10/150] thomas.loc.gov -> vraiLabel=real, original=f

In [1]:
!ls -lh /kaggle/working/

total 0


**TEST COCHRAN**


In [ ]:
def calcul_cochran_q(chemin_csv, nom, col_connu, col_peu_connu, col_anonyme):
    articles = lire_csv(chemin_csv)

    tab=[]
    for a in articles:
        if a[col_connu] == "real":
            connu = 1
        else:
            connu = 0

        if a[col_peu_connu] == "real":
            peu_connu = 1
        else:
            peu_connu = 0

        if a[col_anonyme] == "real":
            anonyme = 1
        else:
            anonyme = 0

        tab.append([connu, peu_connu, anonyme])
        
    resultat = cochrans_q(tab)

    print(f"=== {nom} ===")
    print(f"  Statistique Q : {resultat.statistic:.2f}")
    print(f"  p-value : {resultat.pvalue:.2e}")

    if resultat.pvalue < 0.001:
        print("  => Les 3 conditions different significativement entre elles (p<0.001)")
    else:
        print("  => Pas de difference significative globale")
    print()

    return resultat
    

In [ ]:
chemin_in = "/kaggle/input/datasets/imenec/test-set-qwen/"

resultat_B_avecRAG = calcul_cochran_q(
    chemin_csv=chemin_in + "resultats_contrefactuel_B_1.csv",
    nom="Categorie B (avec RAG)",
    col_connu="verdict_original",
    col_peu_connu="verdict_oppose",
    col_anonyme="verdict_anonyme"
)

resultat_D_avecRAG = calcul_cochran_q(
    chemin_csv=chemin_in + "resultats_contrefactuel_D_1.csv",
    nom="Categorie D (avec RAG)",
    col_connu="verdict_oppose",
    col_peu_connu="verdict_original",
    col_anonyme="verdict_anonyme"
)

resultat_B_sansRAG = calcul_cochran_q(
    chemin_csv=chemin_in + "resultats_contrefactuel_B_WRAG.csv",
    nom="Categorie B (sans RAG)",
    col_connu="verdict_original",
    col_peu_connu="verdict_oppose",
    col_anonyme="verdict_anonyme"
)

resultat_D_sansRAG = calcul_cochran_q(
    chemin_csv=chemin_in + "resultats_contrefactuel_D_WRAG.csv",
    nom="Categorie D (sans RAG)",
    col_connu="verdict_oppose",
    col_peu_connu="verdict_original",
    col_anonyme="verdict_anonyme"
)

**Test mcneamar**

In [ ]:
 
def comparer_deux_colonnes(chemin_csv, nom_colonne_1, nom_colonne_2):
    # cette fonction compare 2 colonnes dun fichier csv avec le test mcNeamar
    # la fonction retourne un dictionnaire avec b, c, p-value,
    # proportions "real" de chaque colonne, diff + IC95%
    # b: nb d'articles "real" (colonne 1) devenus "fake" (colonne 2)
    # c: nb d'articles "fake" (colonne 1) devenus "real" (colonne 2)
    # p_value: probabilite d'observer ce desequilibre entre b et c
    # p_real_colonne_1: proportion d'articles "real" dans la colonne 1
    # p_real_colonne_2: proportion d'articles "real" dans la colonne 2
    # diff_proportions: variation de la proportion de "real" (colonne_2 - colonne_1)
    # ic95_low/high:(95% de confiance) la plus petite et la plus grande valeur probable de la "vraie" difference
 
    col1, col2 = [], []
    with open(chemin_csv, newline="", encoding="utf-8") as f:
        for ligne in csv.DictReader(f):
            col1.append(ligne[nom_colonne_1].strip().lower())
            col2.append(ligne[nom_colonne_2].strip().lower())
 
    n = len(col1)
 
    table = {"fake_fake": 0, "fake_real": 0, "real_fake": 0, "real_real": 0}
    for a, b_ in zip(col1, col2):
        table[f"{a}_{b_}"] += 1
 
    b = table["real_fake"]   # col1=real, col2=fake 
    c = table["fake_real"]   # col1=fake, col2=real
    nbr_flip = b + c
 
    # mcnemar
    if nbr_flip == 0:
        stat, p = 0.0, 1.0
    else:
        stat = (abs(b - c) - 1) ** 2 / nbr_flip
        p = 1 - chi2.cdf(stat, df=1)
 
    # proportions + IC95% de la difference appariee
    p1 = col1.count("real") / n #proportion d'articles "real" dans la colonne 1
    p2 = col2.count("real") / n #proportion d'articles "real" dans la colonne 2
    
    diff = p2 - p1
    
    se = ((b + c) - (b - c) ** 2 / n) ** 0.5 / n
    ic_low, ic_high = diff - 1.96 * se, diff + 1.96 * se
 
    return {
        "colonne_1": nom_colonne_1,
        "colonne_2": nom_colonne_2,
        "b_real_to_fake": b,
        "c_fake_to_real": c,
        "p_value": p,
        "p_real_colonne_1": p1,
        "p_real_colonne_2": p2,
        "diff_proportions": diff,
        "ic95_low": ic_low,
        "ic95_high": ic_high,
    }

In [ ]:
 
def afficher_resultat(res, alpha=0.05/3):
    #Afficher les resultats de comparer_deux_colonnes() de facon lisible
    
    print(f"Comparaison : {res['colonne_1']} vs {res['colonne_2']}")
    print(f"  b (real->fake) = {res['b_real_to_fake']}, c (fake->real) = {res['c_fake_to_real']}, "
          f"changements = {res['b_real_to_fake'] + res['c_fake_to_real']}")
    print(f"  McNemar : p = {res['p_value']:.4g} "
          f"-> {'significatif' if res['p_value'] < alpha else 'non significatif'} (alpha={alpha})")
    print(f"  Proportion 'real' : {res['p_real_colonne_1']*100:.1f}% -> {res['p_real_colonne_2']*100:.1f}%")
    print(f"  Difference = {res['diff_proportions']*100:+.1f} points, "
          f"IC95% = [{res['ic95_low']*100:+.1f} ; {res['ic95_high']*100:+.1f}]")
    print()
 

 

In [ ]:
chemin_in = "/kaggle/input/datasets/imenec/test-stat/resultats_contrefactuel_B.csv"
print("Gemma sur catégorie B avec RAG")
# cat B , original = connu
# connu vs peu_conu
res1 = comparer_deux_colonnes(chemin_in, "verdict_original", "verdict_oppose")
afficher_resultat(res1)

# peu_connu vs anonyme
res2 = comparer_deux_colonnes(chemin_in, "verdict_oppose", "verdict_anonyme")
afficher_resultat(res2)

# connu vs anonyme
res3 = comparer_deux_colonnes(chemin_in, "verdict_original", "verdict_anonyme")
afficher_resultat(res3)

In [ ]:
chemin_in = "/kaggle/input/datasets/imenec/test-stat/resultats_contrefactuel_B_WRAG.csv"
print("Gemma sur catégorie B sans RAG")
# cat B , original = connu
# connu vs peu_conu
res1 = comparer_deux_colonnes(chemin_in, "verdict_original", "verdict_oppose")
afficher_resultat(res1)

# peu_connu vs anonyme
res2 = comparer_deux_colonnes(chemin_in, "verdict_oppose", "verdict_anonyme")
afficher_resultat(res2)

# connu vs anonyme
res3 = comparer_deux_colonnes(chemin_in, "verdict_original", "verdict_anonyme")
afficher_resultat(res3)

In [ ]:
chemin_in = "/kaggle/input/datasets/imenec/test-stat/resultats_contrefactuel_D.csv"
print("Gemma sur catégorie D avec RAG")
# cat D , original = peu connu
# connu vs peu_conu
res1 = comparer_deux_colonnes(chemin_in, "verdict_oppose", "verdict_original")
afficher_resultat(res1)

# peu_connu vs anonyme
res2 = comparer_deux_colonnes(chemin_in, "verdict_original", "verdict_anonyme")
afficher_resultat(res2)

# connu vs anonyme
res3 = comparer_deux_colonnes(chemin_in, "verdict_oppose", "verdict_anonyme")
afficher_resultat(res3)

In [ ]:
chemin_in = "/kaggle/input/datasets/imenec/test-stat/resultats_contrefactuel_D_WRAG.csv"
print("Gemma sur catégorie D sans RAG")
# cat D , original = peu connu
# connu vs peu_conu
res1 = comparer_deux_colonnes(chemin_in, "verdict_oppose", "verdict_original")
afficher_resultat(res1)

# peu_connu vs anonyme
res2 = comparer_deux_colonnes(chemin_in, "verdict_original", "verdict_anonyme")
afficher_resultat(res2)

# connu vs anonyme
res3 = comparer_deux_colonnes(chemin_in, "verdict_oppose", "verdict_anonyme")
afficher_resultat(res3)

In [ ]:
chemin_in = "/kaggle/input/datasets/imenec/test-set-qwen/resultats_contrefactuel_B_1.csv"
print("Qwen sur catégorie B avec RAG")
# cat B , original = connu
# connu vs peu_conu
res1 = comparer_deux_colonnes(chemin_in, "verdict_original", "verdict_oppose")
afficher_resultat(res1)

# peu_connu vs anonyme
res2 = comparer_deux_colonnes(chemin_in, "verdict_oppose", "verdict_anonyme")
afficher_resultat(res2)

# connu vs anonyme
res3 = comparer_deux_colonnes(chemin_in, "verdict_original", "verdict_anonyme")
afficher_resultat(res3)

In [ ]:
chemin_in = "/kaggle/input/datasets/imenec/test-set-qwen/resultats_contrefactuel_B_WRAG.csv"
print("Qwen sur catégorie B sans RAG")
# cat B , original = connu
# connu vs peu_conu
res1 = comparer_deux_colonnes(chemin_in, "verdict_original", "verdict_oppose")
afficher_resultat(res1)

# peu_connu vs anonyme
res2 = comparer_deux_colonnes(chemin_in, "verdict_oppose", "verdict_anonyme")
afficher_resultat(res2)

# connu vs anonyme
res3 = comparer_deux_colonnes(chemin_in, "verdict_original", "verdict_anonyme")
afficher_resultat(res3)

In [ ]:
chemin_in = "/kaggle/input/datasets/imenec/test-set-qwen/resultats_contrefactuel_D_1.csv"
print("Qwen sur catégorie D avec RAG")
# cat D , original = peu connu
# connu vs peu_conu
res1 = comparer_deux_colonnes(chemin_in, "verdict_oppose", "verdict_original")
afficher_resultat(res1)

# peu_connu vs anonyme
res2 = comparer_deux_colonnes(chemin_in, "verdict_original", "verdict_anonyme")
afficher_resultat(res2)

# connu vs anonyme
res3 = comparer_deux_colonnes(chemin_in, "verdict_oppose", "verdict_anonyme")
afficher_resultat(res3)

In [ ]:
chemin_in = "/kaggle/input/datasets/imenec/test-set-qwen/resultats_contrefactuel_D_WRAG.csv"
print("Qwen sur catégorie D sans RAG")
# cat D , original = peu connu
# connu vs peu_conu
res1 = comparer_deux_colonnes(chemin_in, "verdict_oppose", "verdict_original")
afficher_resultat(res1)

# peu_connu vs anonyme
res2 = comparer_deux_colonnes(chemin_in, "verdict_original", "verdict_anonyme")
afficher_resultat(res2)

# connu vs anonyme
res3 = comparer_deux_colonnes(chemin_in, "verdict_oppose", "verdict_anonyme")
afficher_resultat(res3)